# Parse and define tables of "Low Complexity" and "Low Mappability" regions of the H37Rv genome

# Import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#import pickle

%matplotlib inline

In [2]:
import pandas as pd
from tqdm import tqdm
import sys

%matplotlib inline

In [3]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf
import bioframe.vis

#### Set matplotlib text export settings for Adobe Illustrator

In [4]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [5]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1300)

#### Define sets of coord cols for using BioFrame

In [6]:
HmAlnPAF_CoordCols = ("Query_Name", "Query_Start", "Query_End")

PAF_Query_CoordCols = ("Query_Name", "Query_Start", "Query_End")
PAF_Target_CoordCols = ("Target_Name", "Target_Start", "Target_End")

HmReg_CoordCols = ("Chr", "Start", "End")
HHR_CoordCols = ("Chr", "Start", "End")
HmRegion_CoordCols = HmReg_CoordCols


Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")

RE_CoordCols = ("seqname", "start_0based", "end_1based")

GenomeAnno_CoordCols = ("Chrom", "Start", "End")


In [7]:
GenomeAnno_CoordCols

('Chrom', 'Start', 'End')

### Import `gcutils` custom functions

In [8]:
%load_ext autoreload
%autoreload 2
    
from gcutils.general import Rv_dist
from gcutils.general import label_DF_ByOvrLapGenes
from gcutils.general import check_overlap_with_gene_group 


# from gcutils.homologymapfuncs import read_mm2_homology_map_paf, label_HmMap_PAF_DF_ByOvrLapGenes, add_self_overlap_column
# from gcutils.homologymapfuncs import load_and_process_mm2_homology_map_paf, Annotate_HmMap_Aln_DF
# from gcutils.homologymapfuncs import build_ungapped_homology_regions

# Import/parse processed H37rv genome annotations

In [9]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [10]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


## Define relevant H37Rv gene lists for analysis (PE/PPE, Esx, 13E12 gene)

In [11]:
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

In [12]:
listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

In [13]:
listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  


# Define directories for `Low Complexity` Regions + `Low Pileup Mappability` Regions in H37Rv

In [14]:
RepoRef_Dir = "../../References"

Rv_longdust_LCB_OutDir = f"{RepoRef_Dir}/H37Rv_Longdust_LowComplexityRegions"
Rv_LowComplexityRegions_BED = f"{Rv_longdust_LCB_OutDir}/H37Rv.longdust.LCR.relaxedshorter.bed"


Rv_LowPupMap_OutDir = f"{RepoRef_Dir}/H37Rv_MappabilityAnalysis_K50E4"

Rv_LowPupmap_K50E4_BED = f"{Rv_LowPupMap_OutDir}/H37Rv.LowPileupMap.Below1.K50_E4.bed"

In [15]:
!ls -1 $RepoRef_Dir

190927_H37rv_GeneAnnotationsAndLists
190927_H37rv_ListOf_ESXgenes.tsv
201027_H37rv_AnnotatedGenes_And_IntergenicRegions
H37Rv_GenomeWindows
H37Rv_Longdust_LowComplexityRegions
H37Rv_MappabilityAnalysis_K50E4
README.md
WHO_MtbAMR_Catalog


In [16]:
!ls -1 $Rv_longdust_LCB_OutDir

H37Rv.longdust.LCR.default.bed
H37Rv.longdust.LCR.relaxed_e40.bed
H37Rv.longdust.LCR.relaxedshorter.bed
H37Rv.longdust.LCR.relaxedshorter.tsv


In [17]:
!ls -1 $Rv_LowPupMap_OutDir

H37Rv.LowPileupMap.Below1.K50_E4.bed
H37Rv.LowPileupMap.Below1.K50_E4.tsv
H37Rv.PerFeature.MeanPileupMap.K50_E4.tsv.gz
H37Rv.PileupMap.K50_E4.bedgraph.gz
H37Rv.PileupMap.K50_E4.bigwig
H37Rv.PileupMap.K50_E4.Summary.tsv


# Process both BED tables to annotated Bioframe DFs

## a) `H37Rv Low Pileup Mappability` Regions

In [18]:
Rv_LowPmap_DF = bf.read_table(
                  Rv_LowPupmap_K50E4_BED,
                  schema="bed3",
)

Rv_LowPmap_DF["Length"] = Rv_LowPmap_DF["end"] - Rv_LowPmap_DF["start"]

Rv_LowPmap_V2_DF = label_DF_ByOvrLapGenes(
                       Rv_LowPmap_DF,
                       H37Rv_GenomeAnno_Genes_DF,
                       i_cols1 = ("chrom", "start", "end")
                       )


print(Rv_LowPmap_V2_DF.shape)

(1092, 5)


In [19]:
Rv_LowPmap_V2_DF.head(3)

,chrom,start,end,Length,Overlap_Genes
0,NC_000962.3,23172,23237,65,pstP
1,NC_000962.3,79498,79573,75,Rv0071
2,NC_000962.3,80180,80379,199,Rv0071


In [20]:
Rv_LowPmap_V2_DF["Length"].sum()

189147

## a) `H37Rv Low Pileup Mappability` Regions

In [21]:
Rv_LCRs_DF = bf.read_table(
                  Rv_LowComplexityRegions_BED,
                  schema="bed3",
)

Rv_LCRs_DF["Length"] = Rv_LCRs_DF["end"] - Rv_LCRs_DF["start"]

Rv_LCRs_V2_DF = label_DF_ByOvrLapGenes(
                       Rv_LCRs_DF,
                       H37Rv_GenomeAnno_Genes_DF,
                       i_cols1 = ("chrom", "start", "end")
                       )


print(Rv_LCRs_V2_DF.shape)

(231, 5)


In [22]:
Rv_LCRs_V2_DF.head(3)

,chrom,start,end,Length,Overlap_Genes
0,NC_000962.3,7158,7172,14,gyrB
1,NC_000962.3,17896,17904,8,pknA
2,NC_000962.3,18053,18067,14,pknA


In [23]:
Rv_LCRs_V2_DF["Length"].sum()

127160

## Output annotated `LCRs` and `LowPupMap` regions as TSVs (for H37Rv)

In [24]:
RepoRef_Dir = "../../References"
Rv_longdust_LCB_OutDir = f"{RepoRef_Dir}/H37Rv_Longdust_LowComplexityRegions"
Rv_LowPupMap_OutDir = f"{RepoRef_Dir}/H37Rv_MappabilityAnalysis_K50E4"

Rv_LowPupmap_K50E4_TSV = f"{Rv_LowPupMap_OutDir}/H37Rv.LowPileupMap.Below1.K50_E4.tsv"
Rv_LowComplexityRegions_TSV = f"{Rv_longdust_LCB_OutDir}/H37Rv.longdust.LCR.relaxedshorter.tsv"


In [25]:
Rv_LowPmap_V2_DF.to_csv(Rv_LowPupmap_K50E4_TSV, sep ="\t", index=False)

In [26]:
Rv_LCRs_V2_DF.to_csv(Rv_LowComplexityRegions_TSV, sep ="\t", index=False)

In [27]:
!ls -lah $Rv_LowPupmap_K50E4_TSV

-rw-r--r-- 1 mm774 hpc_farhat 41K Mar 10 17:55 ../../References/H37Rv_MappabilityAnalysis_K50E4/H37Rv.LowPileupMap.Below1.K50_E4.tsv


In [28]:
!ls -lah $Rv_LowComplexityRegions_TSV

-rw-r--r-- 1 mm774 hpc_farhat 8.7K Mar 10 17:55 ../../References/H37Rv_Longdust_LowComplexityRegions/H37Rv.longdust.LCR.relaxedshorter.tsv


## Parse `LowComplexity` and `Low-Mappability` Regions of H37Rv

In [29]:
RepoRef_Dir = "../../References"
Rv_longdust_LCB_OutDir = f"{RepoRef_Dir}/H37Rv_Longdust_LowComplexityRegions"
Rv_LowPupMap_OutDir = f"{RepoRef_Dir}/H37Rv_MappabilityAnalysis_K50E4"

Rv_LowPupmap_K50E4_TSV = f"{Rv_LowPupMap_OutDir}/H37Rv.LowPileupMap.Below1.K50_E4.tsv"
Rv_LowComplexityRegions_TSV = f"{Rv_longdust_LCB_OutDir}/H37Rv.longdust.LCR.relaxedshorter.tsv"


In [30]:
Rv_LowPmap_DF = pd.read_csv(Rv_LowPupmap_K50E4_TSV, sep ="\t")
Rv_LowPmap_DF.shape

(1092, 5)

In [31]:
Rv_LCRs_DF = pd.read_csv(Rv_LowComplexityRegions_TSV, sep ="\t")
Rv_LCRs_DF.shape

(231, 5)

# Extras

## Explore overlap of the `LCR` and `LowPmap` regions

In [32]:
Rv_LowPmap_V2_DF.shape

(1092, 5)

In [33]:
Rv_LCRs_V2_DF.shape

(231, 5)

In [34]:
A = bf.overlap(Rv_LowPmap_V2_DF, Rv_LCRs_V2_DF )
A.shape

(1094, 10)

In [35]:
A.shape

(1094, 10)

In [36]:
B = bf.overlap(Rv_LCRs_V2_DF, Rv_LowPmap_V2_DF )
B.shape

(410, 10)

In [37]:
A.head(4)

,chrom,start,end,Length,Overlap_Genes,chrom_,start_,end_,Length_,Overlap_Genes_
0,NC_000962.3,23172,23237,65,pstP,None,<NA>,<NA>,NaN,None
1,NC_000962.3,79498,79573,75,Rv0071,NC_000962.3,79504,79557,53.0,Rv0071
2,NC_000962.3,80180,80379,199,Rv0071,None,<NA>,<NA>,NaN,None
3,NC_000962.3,80458,80517,59,_,None,<NA>,<NA>,NaN,None


In [38]:
C = bf.coverage(Rv_LowPmap_V2_DF, Rv_LCRs_V2_DF)
#C = C[ ( C["coverage"] / (C["end"]-df["start"]) ) >=0.50]

In [39]:
C.shape

(1092, 6)

In [40]:
C.query("coverage > 0").shape

(292, 6)

In [41]:
C.head()

,chrom,start,end,Length,Overlap_Genes,coverage
0,NC_000962.3,23172,23237,65,pstP,0
1,NC_000962.3,79498,79573,75,Rv0071,53
2,NC_000962.3,80180,80379,199,Rv0071,0
3,NC_000962.3,80458,80517,59,_,0
4,NC_000962.3,82165,82237,72,Rv0073,0


In [42]:
D = bf.coverage(Rv_LCRs_V2_DF, Rv_LowPmap_V2_DF)
#C = C[ ( C["coverage"] / (C["end"]-df["start"]) ) >=0.50]

In [43]:
Rv_LCRs_V2_DF.shape

(231, 5)

In [44]:
D.shape

(231, 6)

In [45]:
D.query("coverage > 0").shape

(115, 6)